# 01 — Data Collection

Download and assemble all raw datasets for the Pennsylvania healthcare access analysis.

**Data sources (facilities — hybrid pipeline, pre-geocoded first):**
- HIFLD Hospitals layer (pre-geocoded)
- HRSA Health Center Service Delivery Sites (pre-geocoded)
- HIFLD Urgent Care Facilities (pre-geocoded)
- CMS Provider of Services (CSV gap-fill)
- NPI Registry (uncapped gap-fill of last resort)

**Data sources (tracts & demographics):**
- US Census American Community Survey (ACS) via API
- CDC Social Vulnerability Index (SVI)
- Census TIGER/Line shapefiles (tracts & roads)

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.config import DATA_RAW, STATE_FIPS, CENSUS_API_KEY
from src.data_collection import (
    download_cms_data,
    download_hrsa_data,
    download_urgent_care_data,
    download_cms_pos_data,
    download_npi_gapfill,
    fetch_acs_data,
    download_svi_data,
    download_tiger_shapefiles,
)

## 1.1 Hospitals (HIFLD pre-geocoded, NPI fallback)

Primary source: HIFLD Hospitals layer (ArcGIS FeatureServer, rooftop-accurate coordinates). Falls back to a local CSV in `data/raw/cms/`, then to the uncapped NPI Registry if neither is available.

In [2]:
cms_df = download_cms_data(DATA_RAW)
cms_df.head(), cms_df.shape

Fetching PA hospitals from HRSA gisportal (CMS hospital layers)...


  Hospitals layer returned 235 records


  Critical Access Hospitals layer returned 17 records


(              facility_id                                 facility_name  \
 0  hifld_hospitals_000000            GEISINGER COMMUNITY MEDICAL CENTER   
 1  hifld_hospitals_000001                               UPMC MCKEESPORT   
 2  hifld_hospitals_000002                 GEISINGER-BLOOMSBURG HOSPITAL   
 3  hifld_hospitals_000003  PENN STATE HEALTH HOLY SPIRIT MEDICAL CENTER   
 4  hifld_hospitals_000004                      GEISINGER MEDICAL CENTER   
 
              address        city state    zip   latitude  longitude  \
 0   1822 Mulberry St    Scranton    PA  18510  41.400650 -75.645836   
 1       1500 5th Ave  Mckeesport    PA  15132  40.351719 -79.850373   
 2        549 Fair St  Bloomsburg    PA  17815  41.008754 -76.452758   
 3      503 N 21st St   Camp Hill    PA  17011  40.253433 -76.924041   
 4  100 N Academy Ave    Danville    PA  17822  40.968409 -76.601491   
 
    provider_count           source  
 0           294.0  hifld_hospitals  
 1           198.0  hifld_hospit

## 1.2 HRSA Health Centers (pre-geocoded, NPI fallback)

Primary source: HRSA Health Center Service Delivery and Look-Alike Sites FeatureServer. Falls back to a local CSV in `data/raw/hrsa/`, then to the uncapped NPI Registry if neither is available.

In [3]:
hrsa_df = download_hrsa_data(DATA_RAW)
hrsa_df.head(), hrsa_df.shape

Fetching PA health centers from HRSA service delivery sites layer...


  HRSA returned 378 health center records


(      facility_id                       facility_name             address  \
 0  hrsa_hc_000000                      Lebanon Center       300 Willow St   
 1  hrsa_hc_000001  Schuylkill Community Health Center      210 Sunbury St   
 2  hrsa_hc_000002                      VALLEY MEDICAL    75 S Wyoming Ave   
 3  hrsa_hc_000003             HHC- South Allison Hill       110 S 17th St   
 4  hrsa_hc_000004    FPCSN at Building 21 High School  6501 Limekiln Pike   
 
            city state    zip   latitude  longitude  provider_count   source  
 0       Lebanon    PA  17046  40.341106 -76.417312            60.0  hrsa_hc  
 1   Minersville    PA  17954  40.691090 -76.261030            42.5  hrsa_hc  
 2      Kingston    PA  18704  41.256194 -75.901289            44.0  hrsa_hc  
 3    Harrisburg    PA  17104  40.265634 -76.864267            56.0  hrsa_hc  
 4  Philadelphia    PA  19138  40.053699 -75.152786            47.5  hrsa_hc  ,
 (378, 10))

## 1.3 Urgent Care Facilities (HIFLD)

Primary source: HIFLD Urgent Care Facilities FeatureServer. Falls back to NPI Registry urgent care / ambulatory-surgery taxonomies if the layer is unavailable.

In [4]:
urgent_df = download_urgent_care_data(DATA_RAW)
urgent_df.head(), urgent_df.shape

Fetching PA urgent/ambulatory care facilities from HRSA gisportal...


  Ambulatory Surgical Centers layer returned 235 records


  Rural Health Clinics layer returned 75 records


(                facility_id                              facility_name  \
 0  hifld_urgent_care_000000            PENNSYLVANIA EYE SURGERY CENTER   
 1  hifld_urgent_care_000001                NEI AMBULATORY SURGERY, LLC   
 2  hifld_urgent_care_000002     SOUTHWESTERN AMBULATORY SURGERY CENTER   
 3  hifld_urgent_care_000003  POCONO AMBULATORY SURGERY CENTER, LIMITED   
 4  hifld_urgent_care_000004               APPLE HILL SURGICAL PARTNERS   
 
                        address          city state    zip   latitude  \
 0          4100 Linglestown Rd    Harrisburg    PA  17112  40.335129   
 1              204 Mifflin Ave      Scranton    PA  18503  41.410984   
 2  500 N Lewis Run Rd, Ste 202  West Mifflin    PA  15122  40.328536   
 3                   1 Storm St   Stroudsburg    PA  18360  40.985644   
 4      25 Monument Rd, Ste 270          York    PA  17403  39.923591   
 
    longitude  provider_count             source  
 0 -76.836462             4.0  hifld_urgent_care  
 1 -75

## 1.4 CMS Provider of Services (gap-fill)

CMS POS CSV covers CMS-certified facilities (rural health clinics, ambulatory surgery, dialysis) that HIFLD/HRSA miss. Loaded from `data/raw/cms_pos/` if staged locally; otherwise attempts the direct download URL and returns an empty frame if neither succeeds.

In [5]:
pos_df = download_cms_pos_data(DATA_RAW)
pos_df.head(), pos_df.shape

Attempting to download CMS POS file...


  CMS POS unavailable; continuing without POS gap-fill


(Empty DataFrame
 Columns: [facility_id, facility_name, address, city, state, zip, latitude, longitude, provider_count, source]
 Index: [],
 (0, 10))

## 1.5 NPI Registry (uncapped gap-fill)

Last-resort coverage layer. Runs the full expanded taxonomy list through the NPI Registry API with no per-taxonomy cap, so specialty clinics not present in any of the authoritative registries still land in the merged output.

In [6]:
npi_df = download_npi_gapfill(DATA_RAW)
npi_df.head(), npi_df.shape

Fetching uncapped NPI Registry snapshot (gap-fill tier)...


  Saved 7128 NPI records to C:\Users\saksh\Desktop\DS440-Capstone-Project\data\raw\npi\npi_facilities_standardized.csv


(  facility_id               facility_name            address         city  \
 0  npi_000000            AARUSH MANCHANDA  100 N ACADEMY AVE     DANVILLE   
 1  npi_000001  ABINGTON MEMORIAL HOSPITAL     205 NEWTOWN RD   WARMINSTER   
 2  npi_000002  ABINGTON MEMORIAL HOSPITAL   1200 OLD YORK RD     ABINGTON   
 3  npi_000003  ABINGTON MEMORIAL HOSPITAL    7876 SPRING AVE  ELKINS PARK   
 4  npi_000004  ABINGTON MEMORIAL HOSPITAL   1200 OLD YORK RD     ABINGTON   
 
   state    zip  latitude  longitude  provider_count source  
 0    PA  17822       NaN        NaN             1.0    npi  
 1    PA  18974       NaN        NaN             1.0    npi  
 2    PA  19001       NaN        NaN             1.0    npi  
 3    PA  19027       NaN        NaN             1.0    npi  
 4    PA  19001       NaN        NaN             1.0    npi  ,
 (7128, 10))

## 1.6 Census ACS Demographics

In [7]:
if not CENSUS_API_KEY:
    raise ValueError("CENSUS_API_KEY is missing. Add it to .env or src/config.py")

acs_df = fetch_acs_data(state_fips=STATE_FIPS, year=2022, output_dir=DATA_RAW)
acs_df.head(), acs_df.shape

  Fetched pct_uninsured for 3446 tracts


(                                              NAME  total_population  \
 0  Census Tract 301.01; Adams County; Pennsylvania              2658   
 1  Census Tract 301.03; Adams County; Pennsylvania              2416   
 2  Census Tract 301.04; Adams County; Pennsylvania              3395   
 3     Census Tract 302; Adams County; Pennsylvania              5475   
 4     Census Tract 303; Adams County; Pennsylvania              4412   
 
    median_household_income  insurance_universe  households  \
 0                    80231                2657        1061   
 1                   111603                2416         847   
 2                    81150                3395        1285   
 3                    68363                5451        2035   
 4                    81577                4412        1668   
 
    households_no_vehicle  race_total  race_white state county   tract  \
 0                      6        2658        2465    42    001  030101   
 1                     21       

## 1.7 CDC Social Vulnerability Index

In [8]:
_svi_path = DATA_RAW / "svi" / "SVI_2022_Pennsylvania.csv"
if _svi_path.exists():
    print("SVI data already downloaded — loading from disk.")
    svi_df = pd.read_csv(_svi_path)
else:
    svi_df = download_svi_data(state_fips=STATE_FIPS, output_dir=DATA_RAW)

svi_df.head(), svi_df.shape

SVI data already downloaded — loading from disk.


(   ST         STATE ST_ABBR  STCNTY        COUNTY         FIPS  \
 0  42  Pennsylvania      PA   42001  Adams County  42001030101   
 1  42  Pennsylvania      PA   42001  Adams County  42001030103   
 2  42  Pennsylvania      PA   42001  Adams County  42001030104   
 3  42  Pennsylvania      PA   42001  Adams County  42001030200   
 4  42  Pennsylvania      PA   42001  Adams County  42001030300   
 
                                           LOCATION  AREA_SQMI  E_TOTPOP  \
 0  Census Tract 301.01; Adams County; Pennsylvania  21.149362      2658   
 1  Census Tract 301.03; Adams County; Pennsylvania   6.963834      2416   
 2  Census Tract 301.04; Adams County; Pennsylvania  19.341957      3395   
 3     Census Tract 302; Adams County; Pennsylvania  46.744471      5475   
 4     Census Tract 303; Adams County; Pennsylvania  43.148038      4412   
 
    M_TOTPOP  ...  MP_AIAN  EP_NHPI  MP_NHPI  EP_TWOMORE  MP_TWOMORE  \
 0        14  ...      1.1      0.0      1.1         4.3         3

## 1.8 TIGER/Line Shapefiles (Tracts & Roads)

In [9]:
tiger_data = download_tiger_shapefiles(state_fips=STATE_FIPS, output_dir=DATA_RAW)
tracts_gdf = tiger_data["tracts"]
roads_gdf = tiger_data["roads"]

tracts_gdf.head(2), roads_gdf.head(2), (tracts_gdf.shape, roads_gdf.shape)

TIGER/Line GeoPackages already present; skipping download

(  STATEFP COUNTYFP TRACTCE        GEOID               GEOIDFQ     NAME  \
 0      42      017  100102  42017100102  1400000US42017100102  1001.02   
 1      42      017  100103  42017100103  1400000US42017100103  1001.03   
 
                NAMELSAD  MTFCC FUNCSTAT    ALAND   AWATER     INTPTLAT  \
 0  Census Tract 1001.02  G5020        S  7695740  2422611  +40.0745577   
 1  Census Tract 1001.03  G5020        S  1578123    14537  +40.0682737   
 
        INTPTLON                                           geometry  
 0  -074.9405807  MULTIPOLYGON (((-74.98425 40.0559, -74.98354 4...  
 1  -074.9721495  MULTIPOLYGON (((-74.98519 40.05711, -74.98515 ...  ,
        LINEARID            FULLNAME RTTYP  MTFCC  \
 0  110471840611  Kohler Mill Rd Exd     M  S1400   
 1  110471835209        Ridge Rd Exd     M  S1400   
 
                                             geometry  
 0  LINESTRING (-77.06289 39.85817, -77.06332 39.8...  
 1  LINESTRING (-77.25327 39.76478, -77.25327 39.7...  ,
 ((34

## 1.9 Initial Data Quality Report

In [10]:
quality_report = pd.DataFrame(
    [
        {"dataset": "CMS (HIFLD hospitals)", "rows": len(cms_df), "cols": cms_df.shape[1]},
        {"dataset": "HRSA health centers", "rows": len(hrsa_df), "cols": hrsa_df.shape[1]},
        {"dataset": "Urgent care (HIFLD)", "rows": len(urgent_df), "cols": urgent_df.shape[1]},
        {"dataset": "CMS POS (gap-fill)", "rows": len(pos_df), "cols": pos_df.shape[1]},
        {"dataset": "NPI (gap-fill)", "rows": len(npi_df), "cols": npi_df.shape[1]},
        {"dataset": "ACS", "rows": len(acs_df), "cols": acs_df.shape[1]},
        {"dataset": "SVI", "rows": len(svi_df), "cols": svi_df.shape[1]},
        {"dataset": "Tracts", "rows": len(tracts_gdf), "cols": tracts_gdf.shape[1]},
        {"dataset": "Roads", "rows": len(roads_gdf), "cols": roads_gdf.shape[1]},
    ]
)
quality_report

,dataset,rows,cols
0,CMS (HIFLD hospitals),252,10
1,HRSA health centers,378,10
2,Urgent care (HIFLD),310,10
3,CMS POS (gap-fill),0,10
4,NPI (gap-fill),7128,10
5,ACS,3446,13
6,SVI,3445,161
7,Tracts,3446,14
8,Roads,595975,5
